# Backpropagation

_Deep Learning (CS230)_

**Run the chain rule backward to find how each weight caused the error, then fix it.**

Backpropagation is how a network learns.

     Forward propagation gives a prediction and a loss. Backprop works backward to find how much each weight is to blame for that loss.

---

This notebook is a practice scaffold for the **Backpropagation** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

model = nn.Linear(3, 1)                          # one neuron
opt = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

x = torch.tensor([[1.0, 2.0, 3.0]])              # one input
target = torch.tensor([[10.0]])                  # what we want

pred = model(x)                                  # forward pass
loss = loss_fn(pred, target)                     # how wrong we are

opt.zero_grad()                                  # clear old gradients
loss.backward()                                  # chain rule: fills .grad
print("dL/dw:", model.weight.grad)               # gradient per weight
opt.step()                                       # w <- w - lr * grad
print("loss:", round(loss.item(), 4))

## Visualize the data & results

_When real gradients flow back through a trained digit network, which layers get the biggest weight updates?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier

digits = load_digits()
X, y = digits.data / 16.0, digits.target

net = MLPClassifier(hidden_layer_sizes=(32, 16, 8), max_iter=1, warm_start=True,
                    random_state=1, alpha=1e-3)
net.partial_fit(X, y, classes=np.unique(y))   # one backward pass
before = [w.copy() for w in net.coefs_]
net.partial_fit(X, y)                          # one more step
deltas = [np.abs(a - b).mean() for a, b in zip(net.coefs_, before)]

labels = ["output layer", "hidden 2", "hidden 1", "input layer"]
values = deltas[::-1]                           # output-first
colors = ["#7ee787", "#4ea1ff", "#4ea1ff", "#ff7b72"]

plt.bar(labels, values, color=colors)
plt.title("Real mean weight-update per layer, digit MLP (one SGD step)")
plt.xticks(rotation=15)
plt.show()